In [1]:
# ============================================================
# 07_AURORA_purged_embargoed_walk_forward_models.ipynb
# AURORA-TWETF Purged / Embargoed Walk-Forward Model Evaluation
#
# This notebook combines:
# - 03B: Purged/embargoed baseline model zoo
# - 04C: Purged/embargoed ordinal and uncertainty-aware models
# - 07: Walk-forward purged evaluation
#
# Purpose:
# 1. Load AURORA modeling dataset from Notebook 02.
# 2. Evaluate valid deployable forecasting models only.
# 3. Use target-specific purged/embargoed chronological splits.
# 4. Run expanding-window walk-forward evaluation.
# 5. Compare baseline, ordinal, calibrated, and uncertainty-aware models.
# 6. Save paper-ready metrics, leaderboards, reports, predictions, probabilities,
#    calibration summaries, plots, and manifests.
#
# Important:
# - No previous-label persistence baseline is included in the official comparison.
# - The 20d target uses a 20-row embargo.
# - The 60d target uses a 60-row embargo.
# - All model selection is based on validation folds only.
# ============================================================

from __future__ import annotations

import os
import sys
import json
import math
import time
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

# Optional package installation.
# Uncomment if your Colab runtime does not already have these packages.
# !pip -q install lightgbm xgboost pyarrow joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    log_loss,
    cohen_kappa_score,
    mean_absolute_error,
    mean_squared_error,
)

from joblib import dump

try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False
    print("LightGBM not available. LightGBM models will be skipped.")

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    print("XGBoost not available. XGBoost models will be skipped.")

# ============================================================
# 1. Paths and run configuration
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

MODEL_DATA_PATH = MODELING_DIR / "AURORA_TWETF_features_with_labels.parquet"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "purged_walk_forward_models" / f"run_{RUN_ID}"

PRED_DIR = RUN_ROOT / "predictions"
PROBA_DIR = RUN_ROOT / "probabilities"
MODEL_DIR = RUN_ROOT / "models"
PLOT_DIR = RUN_ROOT / "plots"
CALIBRATION_DIR = RUN_ROOT / "calibration"
REPORT_RUN_DIR = RUN_ROOT / "reports"
DIAGNOSTIC_DIR = RUN_ROOT / "diagnostics"

for d in [
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    RUN_ROOT,
    PRED_DIR,
    PROBA_DIR,
    MODEL_DIR,
    PLOT_DIR,
    CALIBRATION_DIR,
    REPORT_RUN_DIR,
    DIAGNOSTIC_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("AURORA-TWETF Notebook 07: Purged / Embargoed Walk-Forward Models")
print("=" * 80)
print("Timestamp UTC :", RUN_TIMESTAMP)
print("Run ID        :", RUN_ID)
print("Project root  :", PUBLICATION_ROOT)
print("Input data    :", MODEL_DATA_PATH)
print("Run root      :", RUN_ROOT)
print("=" * 80)

if not MODEL_DATA_PATH.exists():
    raise FileNotFoundError(f"Modeling dataset not found: {MODEL_DATA_PATH}")

# ============================================================
# 2. Global modeling configuration
# ============================================================

TARGET_COLS = [
    "TAIEX_regime_fixed_20d",
    "TAIEX_regime_fixed_60d",
]

TARGET_HORIZON_MAP = {
    "TAIEX_regime_fixed_20d": 20,
    "TAIEX_regime_fixed_60d": 60,
}

N_CLASSES = 5
CLASS_LABELS = list(range(N_CLASSES))

REGIME_LABELS = {
    0: "Strong Bear",
    1: "Bear",
    2: "Neutral",
    3: "Bull",
    4: "Strong Bull",
}

RANDOM_STATE = 42

# Minimum split sizes.
MIN_TRAIN_SIZE = 420
MIN_VAL_SIZE = 120
MIN_TEST_SIZE = 120

# Walk-forward folds.
# These are index-ratio anchors. Actual folds will be purged/embargoed.
WALK_FORWARD_FOLDS = [
    {
        "fold_id": "WF1",
        "raw_train_end_ratio": 0.55,
        "raw_val_end_ratio": 0.70,
        "raw_test_end_ratio": 0.85,
    },
    {
        "fold_id": "WF2",
        "raw_train_end_ratio": 0.65,
        "raw_val_end_ratio": 0.80,
        "raw_test_end_ratio": 0.95,
    },
    {
        "fold_id": "WF3",
        "raw_train_end_ratio": 0.70,
        "raw_val_end_ratio": 0.85,
        "raw_test_end_ratio": 1.00,
    },
]

# For speed. Set to False if you want a lighter debugging run.
RUN_TREE_MODELS = True
RUN_BOOSTING_MODELS = True
SAVE_FITTED_MODELS = True

# Probability ensemble top-k from validation.
ENSEMBLE_TOP_K = 5

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
    )

def ensure_datetime_index(df):
    out = df.copy()

    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"])
        out = out.set_index("date")

    out.index = pd.to_datetime(out.index)
    out = out.sort_index()
    out.index.name = "date"

    return out

def proba_cols():
    return [f"proba_class_{i}" for i in range(N_CLASSES)]

def normalize_proba(p):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0

    if np.any(zero_rows):
        p[zero_rows, :] = 1.0 / p.shape[1]
        row_sums = p.sum(axis=1, keepdims=True)

    return p / row_sums

def quadratic_weighted_kappa(y_true, y_pred):
    try:
        return cohen_kappa_score(y_true, y_pred, weights="quadratic", labels=CLASS_LABELS)
    except Exception:
        return np.nan

def ordinal_mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def ordinal_rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def adjacent_accuracy(y_true, y_pred, tol=1):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(np.abs(y_true - y_pred) <= tol))

def multiclass_brier_score(y_true, proba):
    y_true = np.asarray(y_true, dtype=int)
    proba = normalize_proba(proba)

    y_onehot = np.zeros_like(proba)
    valid = (y_true >= 0) & (y_true < proba.shape[1])
    y_onehot[np.arange(len(y_true))[valid], y_true[valid]] = 1.0

    return float(np.mean(np.sum((proba - y_onehot) ** 2, axis=1)))

def expected_calibration_error(y_true, proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    proba = normalize_proba(proba)

    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y_true).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]

        if i == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)

        if mask.sum() == 0:
            continue

        bin_acc = correct[mask].mean()
        bin_conf = conf[mask].mean()
        ece += (mask.mean()) * abs(bin_acc - bin_conf)

    return float(ece)

def probability_diagnostics(proba):
    proba = normalize_proba(proba)

    class_values = np.arange(proba.shape[1], dtype=float)
    expected_class = proba @ class_values
    entropy = -np.sum(np.clip(proba, 1e-12, 1.0) * np.log(np.clip(proba, 1e-12, 1.0)), axis=1)
    normalized_entropy = entropy / np.log(proba.shape[1])
    sorted_p = np.sort(proba, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    ordinal_variance = (proba @ (class_values ** 2)) - expected_class ** 2

    return {
        "expected_class": expected_class,
        "entropy": entropy,
        "normalized_entropy": normalized_entropy,
        "max_probability": proba.max(axis=1),
        "probability_margin": margin,
        "ordinal_variance": ordinal_variance,
    }

def safe_log_loss(y_true, proba):
    try:
        return float(log_loss(y_true, normalize_proba(proba), labels=CLASS_LABELS))
    except Exception:
        return np.nan

def compute_metrics(y_true, y_pred, proba=None):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    row = {}

    row["accuracy"] = float(accuracy_score(y_true, y_pred))
    row["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
    row["macro_f1"] = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    row["weighted_f1"] = float(f1_score(y_true, y_pred, average="weighted", zero_division=0))
    row["macro_precision"] = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    row["macro_recall"] = float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    row["quadratic_weighted_kappa"] = float(quadratic_weighted_kappa(y_true, y_pred))
    row["ordinal_mae"] = float(ordinal_mae(y_true, y_pred))
    row["ordinal_rmse"] = float(ordinal_rmse(y_true, y_pred))
    row["adjacent_accuracy_tol_1"] = float(adjacent_accuracy(y_true, y_pred, tol=1))
    row["large_error_rate_gt_1"] = float(np.mean(np.abs(y_true - y_pred) > 1))
    row["extreme_error_rate_gt_2"] = float(np.mean(np.abs(y_true - y_pred) > 2))

    if proba is not None:
        proba = normalize_proba(proba)
        diag = probability_diagnostics(proba)

        row["multiclass_log_loss"] = safe_log_loss(y_true, proba)
        row["multiclass_brier"] = multiclass_brier_score(y_true, proba)
        row["ece_10bin"] = expected_calibration_error(y_true, proba, n_bins=10)
        row["mean_expected_class"] = float(np.mean(diag["expected_class"]))
        row["mean_max_probability"] = float(np.mean(diag["max_probability"]))
        row["mean_entropy"] = float(np.mean(diag["entropy"]))
        row["mean_normalized_entropy"] = float(np.mean(diag["normalized_entropy"]))
        row["mean_probability_margin"] = float(np.mean(diag["probability_margin"]))
        row["mean_ordinal_variance"] = float(np.mean(diag["ordinal_variance"]))
        row["expected_class_mae"] = float(mean_absolute_error(y_true, diag["expected_class"]))
        row["expected_class_rmse"] = float(math.sqrt(mean_squared_error(y_true, diag["expected_class"])))
    else:
        row["multiclass_log_loss"] = np.nan
        row["multiclass_brier"] = np.nan
        row["ece_10bin"] = np.nan
        row["mean_expected_class"] = np.nan
        row["mean_max_probability"] = np.nan
        row["mean_entropy"] = np.nan
        row["mean_normalized_entropy"] = np.nan
        row["mean_probability_margin"] = np.nan
        row["mean_ordinal_variance"] = np.nan
        row["expected_class_mae"] = np.nan
        row["expected_class_rmse"] = np.nan

    return row

def make_prediction_frame(
    dates,
    y_true,
    y_pred,
    proba,
    target_col,
    fold_id,
    split_name,
    model_name,
    model_family,
):
    dates = pd.DatetimeIndex(dates)
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    proba = normalize_proba(proba)

    df = pd.DataFrame(index=dates)
    df.index.name = "date"
    df["run_id"] = RUN_ID
    df["fold_id"] = fold_id
    df["target_col"] = target_col
    df["split"] = split_name
    df["model_name"] = model_name
    df["model_family"] = model_family
    df["y_true"] = y_true
    df["y_pred"] = y_pred

    diag = probability_diagnostics(proba)
    for k, v in diag.items():
        df[k] = v

    return df

def make_probability_frame(
    dates,
    proba,
    target_col,
    fold_id,
    split_name,
    model_name,
    model_family,
):
    dates = pd.DatetimeIndex(dates)
    proba = normalize_proba(proba)

    df = pd.DataFrame(index=dates)
    df.index.name = "date"
    df["run_id"] = RUN_ID
    df["fold_id"] = fold_id
    df["target_col"] = target_col
    df["split"] = split_name
    df["model_name"] = model_name
    df["model_family"] = model_family

    for i in range(proba.shape[1]):
        df[f"proba_class_{i}"] = proba[:, i]

    return df

# ============================================================
# 4. Custom model wrappers
# ============================================================

class RegressionToOrdinalClassifier(BaseEstimator, ClassifierMixin):
    """
    Fits a regressor on ordinal labels and rounds/clips predictions to classes.
    Probability estimates are produced using a softmax over distance to class centers.
    """

    def __init__(self, regressor=None, temperature=0.75):
        self.regressor = regressor
        self.temperature = temperature

    def fit(self, X, y):
        self.classes_ = np.asarray(CLASS_LABELS)
        self.regressor_ = clone(self.regressor)
        self.regressor_.fit(X, y)
        return self

    def predict_continuous(self, X):
        return np.asarray(self.regressor_.predict(X), dtype=float)

    def predict(self, X):
        z = self.predict_continuous(X)
        y = np.rint(z).astype(int)
        y = np.clip(y, CLASS_LABELS[0], CLASS_LABELS[-1])
        return y

    def predict_proba(self, X):
        z = self.predict_continuous(X).reshape(-1, 1)
        centers = np.asarray(CLASS_LABELS, dtype=float).reshape(1, -1)
        dist2 = (z - centers) ** 2
        logits = -dist2 / max(float(self.temperature), 1e-6)
        logits = logits - logits.max(axis=1, keepdims=True)
        p = np.exp(logits)
        return normalize_proba(p)

class CumulativeOrdinalClassifier(BaseEstimator, ClassifierMixin):
    """
    Cumulative ordinal classifier:
    Fits K-1 binary classifiers for P(y > threshold).
    Converts cumulative probabilities into class probabilities.
    """

    def __init__(self, base_estimator=None):
        self.base_estimator = base_estimator

    def fit(self, X, y):
        self.classes_ = np.asarray(CLASS_LABELS)
        self.thresholds_ = CLASS_LABELS[:-1]
        self.estimators_ = []

        y = np.asarray(y, dtype=int)

        for threshold in self.thresholds_:
            binary_y = (y > threshold).astype(int)

            if len(np.unique(binary_y)) < 2:
                self.estimators_.append(("constant", int(binary_y[0])))
            else:
                est = clone(self.base_estimator)
                est.fit(X, binary_y)
                self.estimators_.append(("model", est))

        return self

    def _predict_gt_probs(self, X):
        probs = []

        for kind, est in self.estimators_:
            if kind == "constant":
                p = np.full(X.shape[0], float(est))
            else:
                if hasattr(est, "predict_proba"):
                    p = est.predict_proba(X)[:, 1]
                else:
                    score = est.decision_function(X)
                    p = 1.0 / (1.0 + np.exp(-score))
            probs.append(p)

        gt = np.vstack(probs).T

        # Enforce monotonicity: P(y>0) >= P(y>1) >= ...
        gt_sorted = np.minimum.accumulate(gt, axis=1)
        return np.clip(gt_sorted, 0.0, 1.0)

    def predict_proba(self, X):
        gt = self._predict_gt_probs(X)

        n = gt.shape[0]
        p = np.zeros((n, N_CLASSES), dtype=float)

        p[:, 0] = 1.0 - gt[:, 0]

        for k in range(1, N_CLASSES - 1):
            p[:, k] = gt[:, k - 1] - gt[:, k]

        p[:, N_CLASSES - 1] = gt[:, N_CLASSES - 2]

        return normalize_proba(p)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

class ValidationTemperatureScaler:
    """
    Simple probability temperature scaler using validation log-loss.
    """

    def __init__(self, candidate_temperatures=None):
        if candidate_temperatures is None:
            candidate_temperatures = np.linspace(0.5, 3.0, 26)
        self.candidate_temperatures = np.asarray(candidate_temperatures, dtype=float)

    def fit(self, y_val, proba_val):
        y_val = np.asarray(y_val, dtype=int)
        proba_val = normalize_proba(proba_val)

        best_t = 1.0
        best_loss = np.inf

        for t in self.candidate_temperatures:
            p = self.transform_proba(proba_val, temperature=t)
            loss = safe_log_loss(y_val, p)

            if np.isfinite(loss) and loss < best_loss:
                best_loss = loss
                best_t = float(t)

        self.temperature_ = best_t
        self.validation_log_loss_ = best_loss
        return self

    def transform_proba(self, proba, temperature=None):
        if temperature is None:
            temperature = getattr(self, "temperature_", 1.0)

        proba = normalize_proba(proba)
        logits = np.log(np.clip(proba, 1e-12, 1.0))
        logits = logits / max(float(temperature), 1e-6)
        logits = logits - logits.max(axis=1, keepdims=True)

        p = np.exp(logits)
        return normalize_proba(p)

class ProbabilityEnsemble:
    """
    Validation-weighted probability ensemble.
    Lower validation composite rank receives higher weight.
    """

    def __init__(self, model_names, weights):
        self.model_names = list(model_names)
        weights = np.asarray(weights, dtype=float)
        if weights.sum() <= 0:
            weights = np.ones_like(weights) / len(weights)
        else:
            weights = weights / weights.sum()
        self.weights = weights

    def combine(self, proba_dict):
        p_final = None

        for name, weight in zip(self.model_names, self.weights):
            p = normalize_proba(proba_dict[name])
            if p_final is None:
                p_final = weight * p
            else:
                p_final += weight * p

        return normalize_proba(p_final)

# ============================================================
# 5. Load and validate data
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading modeling dataset")
print("=" * 80)

df = pd.read_parquet(MODEL_DATA_PATH)
df = ensure_datetime_index(df)

missing_targets = [c for c in TARGET_COLS if c not in df.columns]
if missing_targets:
    raise ValueError(f"Missing target columns: {missing_targets}")

feature_cols = [c for c in df.columns if c not in TARGET_COLS]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]

df = df[feature_cols + TARGET_COLS].copy()
df = df.replace([np.inf, -np.inf], np.nan)

# Drop rows with missing target values.
df = df.dropna(subset=TARGET_COLS)

for target_col in TARGET_COLS:
    df[target_col] = df[target_col].astype(int)

print("Dataset shape        :", df.shape)
print("Feature columns      :", len(feature_cols))
print("Date range           :", df.index.min().date(), "to", df.index.max().date())

for target_col in TARGET_COLS:
    print("\nTarget distribution:", target_col)
    print(df[target_col].value_counts().sort_index().to_string())

dataset_report = {
    "run_id": RUN_ID,
    "model_data_path": str(MODEL_DATA_PATH),
    "dataset_shape": list(df.shape),
    "n_features": len(feature_cols),
    "target_cols": TARGET_COLS,
    "date_start": str(df.index.min().date()),
    "date_end": str(df.index.max().date()),
}

save_json(REPORT_RUN_DIR / "dataset_report.json", dataset_report)

# ============================================================
# 6. Purged/embargoed walk-forward split builder
# ============================================================

def make_purged_walk_forward_splits(df, target_col):
    """
    Creates expanding-window walk-forward splits with target-specific embargo.
    For forward-return horizon h, validation begins h rows after raw_train_end,
    and test begins h rows after raw_val_end.
    """

    n = len(df)
    horizon = int(TARGET_HORIZON_MAP[target_col])
    splits = []

    for spec in WALK_FORWARD_FOLDS:
        fold_id = spec["fold_id"]

        raw_train_end = int(n * spec["raw_train_end_ratio"])
        raw_val_end = int(n * spec["raw_val_end_ratio"])
        raw_test_end = int(n * spec["raw_test_end_ratio"])

        train_start = 0
        train_end = raw_train_end

        val_start = raw_train_end + horizon
        val_end = raw_val_end

        test_start = raw_val_end + horizon
        test_end = raw_test_end

        if train_end <= train_start:
            continue
        if val_end <= val_start:
            continue
        if test_end <= test_start:
            continue

        train_idx = np.arange(train_start, train_end)
        val_idx = np.arange(val_start, val_end)
        test_idx = np.arange(test_start, test_end)

        if len(train_idx) < MIN_TRAIN_SIZE:
            print(f"Skipping {target_col} {fold_id}: train too small ({len(train_idx)})")
            continue
        if len(val_idx) < MIN_VAL_SIZE:
            print(f"Skipping {target_col} {fold_id}: val too small ({len(val_idx)})")
            continue
        if len(test_idx) < MIN_TEST_SIZE:
            print(f"Skipping {target_col} {fold_id}: test too small ({len(test_idx)})")
            continue

        split = {
            "target_col": target_col,
            "fold_id": fold_id,
            "horizon": horizon,
            "raw_train_end_index": raw_train_end,
            "raw_val_end_index": raw_val_end,
            "raw_test_end_index": raw_test_end,
            "train_start_index": int(train_idx[0]),
            "train_end_index": int(train_idx[-1]),
            "val_start_index": int(val_idx[0]),
            "val_end_index": int(val_idx[-1]),
            "test_start_index": int(test_idx[0]),
            "test_end_index": int(test_idx[-1]),
            "train_start_date": str(df.index[train_idx[0]].date()),
            "train_end_date": str(df.index[train_idx[-1]].date()),
            "val_start_date": str(df.index[val_idx[0]].date()),
            "val_end_date": str(df.index[val_idx[-1]].date()),
            "test_start_date": str(df.index[test_idx[0]].date()),
            "test_end_date": str(df.index[test_idx[-1]].date()),
            "n_train": int(len(train_idx)),
            "n_val": int(len(val_idx)),
            "n_test": int(len(test_idx)),
            "embargo_rows": horizon,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
        }

        splits.append(split)

    return splits

all_split_rows = []
splits_by_target = {}

for target_col in TARGET_COLS:
    splits = make_purged_walk_forward_splits(df, target_col)
    splits_by_target[target_col] = splits

    for s in splits:
        row = {k: v for k, v in s.items() if not k.endswith("_idx")}
        all_split_rows.append(row)

split_report_df = pd.DataFrame(all_split_rows)
split_report_df.to_csv(TABLE_DIR / f"table_47_purged_walk_forward_split_report_{RUN_ID}.csv", index=False)
split_report_df.to_csv(RUN_ROOT / "purged_walk_forward_split_report.csv", index=False)

print("\nPurged walk-forward split report:")
print(split_report_df.to_string(index=False))

# ============================================================
# 7. Model factory
# ============================================================

def make_preprocessor(scale=True):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    return steps

def make_model_zoo():
    """
    Valid deployable models only.
    No previous-label persistence is included.
    """

    models = {}

    # Dummy baselines.
    models["M0_dummy_most_frequent"] = {
        "family": "baseline_dummy",
        "estimator": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyClassifier(strategy="most_frequent")),
        ]),
    }

    models["M0_dummy_stratified"] = {
        "family": "baseline_dummy",
        "estimator": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)),
        ]),
    }

    # Multiclass baselines.
    models["M1_logistic_regression_balanced"] = {
        "family": "baseline_multiclass",
        "estimator": Pipeline([
            *make_preprocessor(scale=True),
            ("model", LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                multi_class="auto",
                solver="lbfgs",
                random_state=RANDOM_STATE,
            )),
        ]),
    }

    if RUN_TREE_MODELS:
        models["M2_random_forest_balanced"] = {
            "family": "baseline_multiclass",
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RandomForestClassifier(
                    n_estimators=300,
                    max_depth=6,
                    min_samples_leaf=10,
                    class_weight="balanced_subsample",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )),
            ]),
        }

        models["M3_extra_trees_balanced"] = {
            "family": "baseline_multiclass",
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", ExtraTreesClassifier(
                    n_estimators=300,
                    max_depth=6,
                    min_samples_leaf=10,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )),
            ]),
        }

        models["M4_hist_gradient_boosting"] = {
            "family": "baseline_multiclass",
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", HistGradientBoostingClassifier(
                    max_iter=200,
                    learning_rate=0.04,
                    max_leaf_nodes=15,
                    l2_regularization=1.0,
                    random_state=RANDOM_STATE,
                )),
            ]),
        }

    if RUN_BOOSTING_MODELS and HAS_LGBM:
        models["M5_lightgbm_balanced"] = {
            "family": "baseline_multiclass",
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", lgb.LGBMClassifier(
                    objective="multiclass",
                    num_class=N_CLASSES,
                    n_estimators=300,
                    learning_rate=0.03,
                    max_depth=4,
                    num_leaves=15,
                    min_child_samples=20,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    verbose=-1,
                )),
            ]),
        }

    if RUN_BOOSTING_MODELS and HAS_XGB:
        models["M6_xgboost_multiclass"] = {
            "family": "baseline_multiclass",
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", xgb.XGBClassifier(
                    objective="multi:softprob",
                    num_class=N_CLASSES,
                    n_estimators=300,
                    learning_rate=0.03,
                    max_depth=3,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_lambda=2.0,
                    reg_alpha=0.1,
                    eval_metric="mlogloss",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                )),
            ]),
        }

    # Ordinal cumulative models.
    base_logit = Pipeline([
        *make_preprocessor(scale=True),
        ("model", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            solver="lbfgs",
            random_state=RANDOM_STATE,
        )),
    ])

    models["O1_cumulative_logit_balanced"] = {
        "family": "ordinal_cumulative",
        "estimator": CumulativeOrdinalClassifier(base_estimator=base_logit),
    }

    if RUN_TREE_MODELS:
        base_rf = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=250,
                max_depth=5,
                min_samples_leaf=10,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])

        base_et = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                n_estimators=250,
                max_depth=5,
                min_samples_leaf=10,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])

        models["O2_cumulative_rf_balanced"] = {
            "family": "ordinal_cumulative",
            "estimator": CumulativeOrdinalClassifier(base_estimator=base_rf),
        }

        models["O3_cumulative_et_balanced"] = {
            "family": "ordinal_cumulative",
            "estimator": CumulativeOrdinalClassifier(base_estimator=base_et),
        }

    if RUN_BOOSTING_MODELS and HAS_LGBM:
        base_lgbm_binary = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", lgb.LGBMClassifier(
                objective="binary",
                n_estimators=250,
                learning_rate=0.03,
                max_depth=3,
                num_leaves=7,
                min_child_samples=20,
                subsample=0.8,
                colsample_bytree=0.8,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                verbose=-1,
            )),
        ])

        models["O4_cumulative_lgbm_balanced"] = {
            "family": "ordinal_cumulative",
            "estimator": CumulativeOrdinalClassifier(base_estimator=base_lgbm_binary),
        }

    # Regression-to-ordinal models.
    models["R1_ridge_regression_to_ordinal"] = {
        "family": "regression_to_ordinal",
        "estimator": Pipeline([
            *make_preprocessor(scale=True),
            ("model", RegressionToOrdinalClassifier(
                regressor=Ridge(alpha=10.0, random_state=RANDOM_STATE),
                temperature=0.75,
            )),
        ]),
    }

    if RUN_TREE_MODELS:
        models["R2_rf_regression_to_ordinal"] = {
            "family": "regression_to_ordinal",
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RegressionToOrdinalClassifier(
                    regressor=RandomForestRegressor(
                        n_estimators=250,
                        max_depth=5,
                        min_samples_leaf=10,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                    temperature=0.80,
                )),
            ]),
        }

        models["R3_et_regression_to_ordinal"] = {
            "family": "regression_to_ordinal",
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RegressionToOrdinalClassifier(
                    regressor=ExtraTreesRegressor(
                        n_estimators=250,
                        max_depth=5,
                        min_samples_leaf=10,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                    temperature=0.80,
                )),
            ]),
        }

    if RUN_BOOSTING_MODELS and HAS_LGBM:
        models["R4_lgbm_regression_to_ordinal"] = {
            "family": "regression_to_ordinal",
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RegressionToOrdinalClassifier(
                    regressor=lgb.LGBMRegressor(
                        objective="regression",
                        n_estimators=250,
                        learning_rate=0.03,
                        max_depth=3,
                        num_leaves=7,
                        min_child_samples=20,
                        subsample=0.8,
                        colsample_bytree=0.8,
                        random_state=RANDOM_STATE,
                        verbose=-1,
                    ),
                    temperature=0.80,
                )),
            ]),
        }

    if RUN_BOOSTING_MODELS and HAS_XGB:
        models["R5_xgb_regression_to_ordinal"] = {
            "family": "regression_to_ordinal",
            "estimator": Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RegressionToOrdinalClassifier(
                    regressor=xgb.XGBRegressor(
                        objective="reg:squarederror",
                        n_estimators=250,
                        learning_rate=0.03,
                        max_depth=3,
                        subsample=0.8,
                        colsample_bytree=0.8,
                        reg_lambda=2.0,
                        reg_alpha=0.1,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                    temperature=0.80,
                )),
            ]),
        }

    # Calibrated multiclass models.
    base_cal_logit = Pipeline([
        *make_preprocessor(scale=True),
        ("model", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            multi_class="auto",
            solver="lbfgs",
            random_state=RANDOM_STATE,
        )),
    ])

    models["C1_calibrated_logistic_balanced"] = {
        "family": "calibrated_multiclass",
        "estimator": CalibratedClassifierCV(
            estimator=base_cal_logit,
            method="sigmoid",
            cv=3,
        ),
    }

    if RUN_TREE_MODELS:
        base_cal_rf = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=250,
                max_depth=5,
                min_samples_leaf=10,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])

        models["C2_calibrated_rf_balanced"] = {
            "family": "calibrated_multiclass",
            "estimator": CalibratedClassifierCV(
                estimator=base_cal_rf,
                method="sigmoid",
                cv=3,
            ),
        }

    if RUN_BOOSTING_MODELS and HAS_LGBM:
        base_cal_lgbm = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", lgb.LGBMClassifier(
                objective="multiclass",
                num_class=N_CLASSES,
                n_estimators=250,
                learning_rate=0.03,
                max_depth=3,
                num_leaves=7,
                min_child_samples=20,
                subsample=0.8,
                colsample_bytree=0.8,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                verbose=-1,
            )),
        ])

        models["C3_calibrated_lgbm_balanced"] = {
            "family": "calibrated_multiclass",
            "estimator": CalibratedClassifierCV(
                estimator=base_cal_lgbm,
                method="sigmoid",
                cv=3,
            ),
        }

    if RUN_BOOSTING_MODELS and HAS_XGB:
        base_cal_xgb = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", xgb.XGBClassifier(
                objective="multi:softprob",
                num_class=N_CLASSES,
                n_estimators=250,
                learning_rate=0.03,
                max_depth=3,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=2.0,
                reg_alpha=0.1,
                eval_metric="mlogloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])

        models["C4_calibrated_xgb_multiclass"] = {
            "family": "calibrated_multiclass",
            "estimator": CalibratedClassifierCV(
                estimator=base_cal_xgb,
                method="sigmoid",
                cv=3,
            ),
        }

    return models

def get_model_proba(model, X):
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
    else:
        y_pred = model.predict(X)
        p = np.zeros((len(y_pred), N_CLASSES))
        p[np.arange(len(y_pred)), y_pred.astype(int)] = 1.0

    p = np.asarray(p, dtype=float)

    # Some calibrated models may omit classes if a fold lacks a class.
    if p.shape[1] != N_CLASSES:
        full = np.zeros((p.shape[0], N_CLASSES), dtype=float)
        if hasattr(model, "classes_"):
            classes = list(model.classes_)
            for j, c in enumerate(classes):
                if int(c) in CLASS_LABELS:
                    full[:, int(c)] = p[:, j]
            p = full
        else:
            raise ValueError(f"Probability shape mismatch: {p.shape}")

    return normalize_proba(p)

# ============================================================
# 8. Main training and evaluation loop
# ============================================================

all_metrics_rows = []
all_classification_report_rows = []
all_prediction_frames = []
all_probability_frames = []
all_calibration_rows = []
all_confusion_rows = []
fit_time_rows = []

all_validation_probas = {}
all_test_probas = {}
all_validation_y = {}
all_test_y = {}
all_validation_dates = {}
all_test_dates = {}

print("\n" + "=" * 80)
print("Step 2: Running purged walk-forward model evaluation")
print("=" * 80)

X_all = df[feature_cols].copy()

for target_col in TARGET_COLS:
    print("\n" + "#" * 80)
    print("Target:", target_col)
    print("#" * 80)

    y_all = df[target_col].astype(int).copy()
    splits = splits_by_target[target_col]

    for split in splits:
        fold_id = split["fold_id"]
        horizon = split["horizon"]

        print("\n" + "-" * 80)
        print(f"Target={target_col} | Fold={fold_id} | Horizon={horizon} | Embargo={horizon}")
        print(
            "Train:",
            split["train_start_date"],
            "to",
            split["train_end_date"],
            f"n={split['n_train']}",
        )
        print(
            "Val  :",
            split["val_start_date"],
            "to",
            split["val_end_date"],
            f"n={split['n_val']}",
        )
        print(
            "Test :",
            split["test_start_date"],
            "to",
            split["test_end_date"],
            f"n={split['n_test']}",
        )
        print("-" * 80)

        train_idx = split["train_idx"]
        val_idx = split["val_idx"]
        test_idx = split["test_idx"]

        X_train = X_all.iloc[train_idx]
        X_val = X_all.iloc[val_idx]
        X_test = X_all.iloc[test_idx]

        y_train = y_all.iloc[train_idx].values
        y_val = y_all.iloc[val_idx].values
        y_test = y_all.iloc[test_idx].values

        dates_train = df.index[train_idx]
        dates_val = df.index[val_idx]
        dates_test = df.index[test_idx]

        fold_key = (target_col, fold_id)
        all_validation_y[fold_key] = y_val
        all_test_y[fold_key] = y_test
        all_validation_dates[fold_key] = dates_val
        all_test_dates[fold_key] = dates_test
        all_validation_probas[fold_key] = {}
        all_test_probas[fold_key] = {}

        model_zoo = make_model_zoo()

        for model_name, model_info in model_zoo.items():
            model_family = model_info["family"]
            estimator = clone(model_info["estimator"])

            print(f"Training {model_name} [{model_family}] ...", end=" ")

            t0 = time.time()

            try:
                estimator.fit(X_train, y_train)

                proba_train = get_model_proba(estimator, X_train)
                proba_val = get_model_proba(estimator, X_val)
                proba_test = get_model_proba(estimator, X_test)

                y_pred_train = proba_train.argmax(axis=1)
                y_pred_val = proba_val.argmax(axis=1)
                y_pred_test = proba_test.argmax(axis=1)

                fit_seconds = time.time() - t0
                print(f"done in {fit_seconds:.1f}s")

            except Exception as e:
                fit_seconds = time.time() - t0
                print(f"FAILED after {fit_seconds:.1f}s: {repr(e)}")

                fit_time_rows.append({
                    "run_id": RUN_ID,
                    "target_col": target_col,
                    "fold_id": fold_id,
                    "model_name": model_name,
                    "model_family": model_family,
                    "fit_seconds": fit_seconds,
                    "status": "failed",
                    "error": repr(e),
                })
                continue

            fit_time_rows.append({
                "run_id": RUN_ID,
                "target_col": target_col,
                "fold_id": fold_id,
                "model_name": model_name,
                "model_family": model_family,
                "fit_seconds": fit_seconds,
                "status": "success",
                "error": "",
            })

            all_validation_probas[fold_key][model_name] = proba_val
            all_test_probas[fold_key][model_name] = proba_test

            # Temperature scaling fitted on validation for diagnostic calibrated probabilities.
            scaler = ValidationTemperatureScaler().fit(y_val, proba_val)
            proba_val_temp = scaler.transform_proba(proba_val)
            proba_test_temp = scaler.transform_proba(proba_test)

            # Save raw model metrics and temp-scaled variant.
            split_payloads = [
                ("train", dates_train, y_train, y_pred_train, proba_train),
                ("validation", dates_val, y_val, y_pred_val, proba_val),
                ("test", dates_test, y_test, y_pred_test, proba_test),
            ]

            for split_name, dates_split, y_true_split, y_pred_split, proba_split in split_payloads:
                metric_row = compute_metrics(y_true_split, y_pred_split, proba_split)

                metric_row.update({
                    "run_id": RUN_ID,
                    "target_col": target_col,
                    "horizon": horizon,
                    "fold_id": fold_id,
                    "split": split_name,
                    "model_name": model_name,
                    "model_family": model_family,
                    "calibration_variant": "native",
                    "n_obs": int(len(y_true_split)),
                    "date_start": str(pd.DatetimeIndex(dates_split).min().date()),
                    "date_end": str(pd.DatetimeIndex(dates_split).max().date()),
                    "embargo_rows": horizon,
                })

                all_metrics_rows.append(metric_row)

                pred_frame = make_prediction_frame(
                    dates=dates_split,
                    y_true=y_true_split,
                    y_pred=y_pred_split,
                    proba=proba_split,
                    target_col=target_col,
                    fold_id=fold_id,
                    split_name=split_name,
                    model_name=model_name,
                    model_family=model_family,
                )

                proba_frame = make_probability_frame(
                    dates=dates_split,
                    proba=proba_split,
                    target_col=target_col,
                    fold_id=fold_id,
                    split_name=split_name,
                    model_name=model_name,
                    model_family=model_family,
                )

                all_prediction_frames.append(pred_frame)
                all_probability_frames.append(proba_frame)

                report = classification_report(
                    y_true_split,
                    y_pred_split,
                    labels=CLASS_LABELS,
                    output_dict=True,
                    zero_division=0,
                )

                for label_key, label_metrics in report.items():
                    if isinstance(label_metrics, dict):
                        row = {
                            "run_id": RUN_ID,
                            "target_col": target_col,
                            "horizon": horizon,
                            "fold_id": fold_id,
                            "split": split_name,
                            "model_name": model_name,
                            "model_family": model_family,
                            "label": label_key,
                        }
                        row.update(label_metrics)
                        all_classification_report_rows.append(row)

                cm = confusion_matrix(y_true_split, y_pred_split, labels=CLASS_LABELS)
                for i, true_class in enumerate(CLASS_LABELS):
                    for j, pred_class in enumerate(CLASS_LABELS):
                        all_confusion_rows.append({
                            "run_id": RUN_ID,
                            "target_col": target_col,
                            "horizon": horizon,
                            "fold_id": fold_id,
                            "split": split_name,
                            "model_name": model_name,
                            "model_family": model_family,
                            "true_class": true_class,
                            "pred_class": pred_class,
                            "count": int(cm[i, j]),
                        })

            # Add temperature-scaled validation/test metric rows.
            for split_name, dates_split, y_true_split, proba_split in [
                ("validation", dates_val, y_val, proba_val_temp),
                ("test", dates_test, y_test, proba_test_temp),
            ]:
                y_pred_temp = proba_split.argmax(axis=1)
                metric_row = compute_metrics(y_true_split, y_pred_temp, proba_split)

                metric_row.update({
                    "run_id": RUN_ID,
                    "target_col": target_col,
                    "horizon": horizon,
                    "fold_id": fold_id,
                    "split": split_name,
                    "model_name": model_name + "_temp_scaled",
                    "model_family": model_family + "_temperature_scaled",
                    "calibration_variant": "temperature_scaled",
                    "n_obs": int(len(y_true_split)),
                    "date_start": str(pd.DatetimeIndex(dates_split).min().date()),
                    "date_end": str(pd.DatetimeIndex(dates_split).max().date()),
                    "embargo_rows": horizon,
                    "temperature": float(scaler.temperature_),
                    "temperature_validation_log_loss": float(scaler.validation_log_loss_),
                })

                all_metrics_rows.append(metric_row)

                all_calibration_rows.append({
                    "run_id": RUN_ID,
                    "target_col": target_col,
                    "horizon": horizon,
                    "fold_id": fold_id,
                    "model_name": model_name,
                    "model_family": model_family,
                    "temperature": float(scaler.temperature_),
                    "validation_log_loss_native": safe_log_loss(y_val, proba_val),
                    "validation_log_loss_temp_scaled": safe_log_loss(y_val, proba_val_temp),
                    "validation_ece_native": expected_calibration_error(y_val, proba_val, n_bins=10),
                    "validation_ece_temp_scaled": expected_calibration_error(y_val, proba_val_temp, n_bins=10),
                })

            if SAVE_FITTED_MODELS:
                model_path = MODEL_DIR / f"model_{safe_name(target_col)}_{safe_name(fold_id)}_{safe_name(model_name)}.joblib"
                try:
                    dump(estimator, model_path)
                except Exception as e:
                    print(f"Could not save model {model_name}: {repr(e)}")

# ============================================================
# 9. Build validation-weighted ensembles per target/fold
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Building validation-weighted probability ensembles")
print("=" * 80)

metrics_df_temp = pd.DataFrame(all_metrics_rows)

def validation_composite_leaderboard(metrics_df):
    dfm = metrics_df[
        (metrics_df["split"] == "validation")
        & (metrics_df["calibration_variant"] == "native")
    ].copy()

    if dfm.empty:
        return pd.DataFrame()

    out = []

    for (target_col, fold_id), tmp in dfm.groupby(["target_col", "fold_id"]):
        tmp = tmp.copy()

        tmp["rank_macro_f1"] = tmp["macro_f1"].rank(ascending=False, method="min")
        tmp["rank_balanced_accuracy"] = tmp["balanced_accuracy"].rank(ascending=False, method="min")
        tmp["rank_qwk"] = tmp["quadratic_weighted_kappa"].rank(ascending=False, method="min")
        tmp["rank_ordinal_mae"] = tmp["ordinal_mae"].rank(ascending=True, method="min")
        tmp["rank_ece"] = tmp["ece_10bin"].rank(ascending=True, method="min")

        tmp["validation_composite_rank"] = (
            tmp["rank_macro_f1"]
            + tmp["rank_balanced_accuracy"]
            + tmp["rank_qwk"]
            + tmp["rank_ordinal_mae"]
            + tmp["rank_ece"]
        ) / 5.0

        tmp = tmp.sort_values(
            ["validation_composite_rank", "macro_f1", "balanced_accuracy", "quadratic_weighted_kappa"],
            ascending=[True, False, False, False],
        )

        out.append(tmp)

    return pd.concat(out, ignore_index=True)

val_leader_tmp = validation_composite_leaderboard(metrics_df_temp)

ensemble_metric_rows = []
ensemble_prediction_frames = []
ensemble_probability_frames = []

if not val_leader_tmp.empty:
    for (target_col, fold_id), tmp in val_leader_tmp.groupby(["target_col", "fold_id"]):
        fold_key = (target_col, fold_id)

        available_names = list(all_validation_probas.get(fold_key, {}).keys())
        tmp = tmp[tmp["model_name"].isin(available_names)].copy()

        if tmp.empty:
            continue

        top = tmp.head(ENSEMBLE_TOP_K).copy()

        # Convert rank to inverse rank weights.
        weights = 1.0 / np.maximum(top["validation_composite_rank"].values.astype(float), 1e-6)
        model_names = top["model_name"].tolist()

        ensemble = ProbabilityEnsemble(model_names=model_names, weights=weights)

        val_proba_dict = all_validation_probas[fold_key]
        test_proba_dict = all_test_probas[fold_key]

        p_val_ens = ensemble.combine(val_proba_dict)
        p_test_ens = ensemble.combine(test_proba_dict)

        y_val = all_validation_y[fold_key]
        y_test = all_test_y[fold_key]
        dates_val = all_validation_dates[fold_key]
        dates_test = all_test_dates[fold_key]

        for split_name, dates_split, y_true_split, p_split in [
            ("validation", dates_val, y_val, p_val_ens),
            ("test", dates_test, y_test, p_test_ens),
        ]:
            y_pred_split = p_split.argmax(axis=1)
            metric_row = compute_metrics(y_true_split, y_pred_split, p_split)

            metric_row.update({
                "run_id": RUN_ID,
                "target_col": target_col,
                "horizon": TARGET_HORIZON_MAP[target_col],
                "fold_id": fold_id,
                "split": split_name,
                "model_name": "E1_validation_weighted_probability_ensemble",
                "model_family": "probability_ensemble",
                "calibration_variant": "native",
                "n_obs": int(len(y_true_split)),
                "date_start": str(pd.DatetimeIndex(dates_split).min().date()),
                "date_end": str(pd.DatetimeIndex(dates_split).max().date()),
                "embargo_rows": TARGET_HORIZON_MAP[target_col],
                "ensemble_top_k": ENSEMBLE_TOP_K,
                "ensemble_members": "|".join(model_names),
            })

            ensemble_metric_rows.append(metric_row)

            ensemble_prediction_frames.append(
                make_prediction_frame(
                    dates=dates_split,
                    y_true=y_true_split,
                    y_pred=y_pred_split,
                    proba=p_split,
                    target_col=target_col,
                    fold_id=fold_id,
                    split_name=split_name,
                    model_name="E1_validation_weighted_probability_ensemble",
                    model_family="probability_ensemble",
                )
            )

            ensemble_probability_frames.append(
                make_probability_frame(
                    dates=dates_split,
                    proba=p_split,
                    target_col=target_col,
                    fold_id=fold_id,
                    split_name=split_name,
                    model_name="E1_validation_weighted_probability_ensemble",
                    model_family="probability_ensemble",
                )
            )

        print(
            f"Ensemble built: target={target_col}, fold={fold_id}, members={model_names}"
        )

all_metrics_rows.extend(ensemble_metric_rows)
all_prediction_frames.extend(ensemble_prediction_frames)
all_probability_frames.extend(ensemble_probability_frames)

# ============================================================
# 10. Save core outputs
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Saving model evaluation outputs")
print("=" * 80)

metrics_df = pd.DataFrame(all_metrics_rows)
classification_reports_df = pd.DataFrame(all_classification_report_rows)
confusion_df = pd.DataFrame(all_confusion_rows)
calibration_df = pd.DataFrame(all_calibration_rows)
fit_time_df = pd.DataFrame(fit_time_rows)

predictions_all_df = pd.concat(all_prediction_frames, axis=0).sort_index()
probabilities_all_df = pd.concat(all_probability_frames, axis=0).sort_index()

metrics_path = RUN_ROOT / "purged_walk_forward_metrics_all.csv"
metrics_df.to_csv(metrics_path, index=False)
metrics_df.to_csv(TABLE_DIR / f"table_48_purged_walk_forward_metrics_all_{RUN_ID}.csv", index=False)

classification_reports_df.to_csv(RUN_ROOT / "purged_walk_forward_classification_reports.csv", index=False)
classification_reports_df.to_csv(TABLE_DIR / f"table_49_purged_walk_forward_classification_reports_{RUN_ID}.csv", index=False)

confusion_df.to_csv(RUN_ROOT / "purged_walk_forward_confusion_counts.csv", index=False)
confusion_df.to_csv(TABLE_DIR / f"table_50_purged_walk_forward_confusion_counts_{RUN_ID}.csv", index=False)

calibration_df.to_csv(CALIBRATION_DIR / "temperature_scaling_calibration_summary.csv", index=False)
calibration_df.to_csv(TABLE_DIR / f"table_51_temperature_scaling_calibration_summary_{RUN_ID}.csv", index=False)

fit_time_df.to_csv(RUN_ROOT / "purged_walk_forward_fit_times.csv", index=False)
fit_time_df.to_csv(TABLE_DIR / f"table_52_purged_walk_forward_fit_times_{RUN_ID}.csv", index=False)

predictions_all_df.to_parquet(PRED_DIR / "predictions_all_purged_walk_forward_models.parquet")
predictions_all_df.to_csv(PRED_DIR / "predictions_all_purged_walk_forward_models.csv")

probabilities_all_df.to_parquet(PROBA_DIR / "probabilities_all_purged_walk_forward_models.parquet")
probabilities_all_df.to_csv(PROBA_DIR / "probabilities_all_purged_walk_forward_models.csv")

print("Metrics saved to       :", metrics_path)
print("Predictions saved to   :", PRED_DIR)
print("Probabilities saved to :", PROBA_DIR)

# ============================================================
# 11. Aggregate leaderboards
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Creating aggregate leaderboards")
print("=" * 80)

def make_fold_leaderboard(metrics_df, split_name="validation"):
    base = metrics_df[
        (metrics_df["split"] == split_name)
        & (metrics_df["calibration_variant"] == "native")
    ].copy()

    if base.empty:
        return pd.DataFrame()

    out = []

    for (target_col, fold_id), tmp in base.groupby(["target_col", "fold_id"]):
        tmp = tmp.copy()

        tmp["rank_macro_f1"] = tmp["macro_f1"].rank(ascending=False, method="min")
        tmp["rank_balanced_accuracy"] = tmp["balanced_accuracy"].rank(ascending=False, method="min")
        tmp["rank_qwk"] = tmp["quadratic_weighted_kappa"].rank(ascending=False, method="min")
        tmp["rank_ordinal_mae"] = tmp["ordinal_mae"].rank(ascending=True, method="min")
        tmp["rank_ece"] = tmp["ece_10bin"].rank(ascending=True, method="min")

        tmp["composite_rank"] = (
            tmp["rank_macro_f1"]
            + tmp["rank_balanced_accuracy"]
            + tmp["rank_qwk"]
            + tmp["rank_ordinal_mae"]
            + tmp["rank_ece"]
        ) / 5.0

        tmp = tmp.sort_values(
            ["composite_rank", "macro_f1", "balanced_accuracy", "quadratic_weighted_kappa", "ordinal_mae"],
            ascending=[True, False, False, False, True],
        )

        tmp.insert(0, "leaderboard_scope", f"{target_col}_{fold_id}_{split_name}")
        out.append(tmp)

    return pd.concat(out, ignore_index=True)

leaderboard_val_df = make_fold_leaderboard(metrics_df, split_name="validation")
leaderboard_test_df = make_fold_leaderboard(metrics_df, split_name="test")

leaderboard_val_df.to_csv(RUN_ROOT / "leaderboard_validation_by_fold.csv", index=False)
leaderboard_test_df.to_csv(RUN_ROOT / "leaderboard_test_by_fold.csv", index=False)

leaderboard_val_df.to_csv(TABLE_DIR / f"table_53_leaderboard_validation_by_fold_{RUN_ID}.csv", index=False)
leaderboard_test_df.to_csv(TABLE_DIR / f"table_54_leaderboard_test_by_fold_{RUN_ID}.csv", index=False)

# Aggregate across folds.
agg_cols = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "quadratic_weighted_kappa",
    "ordinal_mae",
    "ordinal_rmse",
    "adjacent_accuracy_tol_1",
    "multiclass_log_loss",
    "multiclass_brier",
    "ece_10bin",
    "mean_normalized_entropy",
    "expected_class_mae",
]

agg_metrics_df = (
    metrics_df[
        (metrics_df["split"].isin(["validation", "test"]))
        & (metrics_df["calibration_variant"] == "native")
    ]
    .groupby(["target_col", "split", "model_name", "model_family"], as_index=False)
    .agg(
        **{f"mean_{c}": (c, "mean") for c in agg_cols},
        **{f"std_{c}": (c, "std") for c in agg_cols},
        n_folds=("fold_id", "nunique"),
        n_total_obs=("n_obs", "sum"),
    )
)

# Composite rank on aggregate test metrics.
agg_leaderboards = []

for (target_col, split_name), tmp in agg_metrics_df.groupby(["target_col", "split"]):
    tmp = tmp.copy()

    tmp["rank_mean_macro_f1"] = tmp["mean_macro_f1"].rank(ascending=False, method="min")
    tmp["rank_mean_balanced_accuracy"] = tmp["mean_balanced_accuracy"].rank(ascending=False, method="min")
    tmp["rank_mean_qwk"] = tmp["mean_quadratic_weighted_kappa"].rank(ascending=False, method="min")
    tmp["rank_mean_ordinal_mae"] = tmp["mean_ordinal_mae"].rank(ascending=True, method="min")
    tmp["rank_mean_ece"] = tmp["mean_ece_10bin"].rank(ascending=True, method="min")

    tmp["aggregate_composite_rank"] = (
        tmp["rank_mean_macro_f1"]
        + tmp["rank_mean_balanced_accuracy"]
        + tmp["rank_mean_qwk"]
        + tmp["rank_mean_ordinal_mae"]
        + tmp["rank_mean_ece"]
    ) / 5.0

    tmp = tmp.sort_values(
        ["aggregate_composite_rank", "mean_macro_f1", "mean_balanced_accuracy", "mean_quadratic_weighted_kappa"],
        ascending=[True, False, False, False],
    )

    tmp.insert(0, "leaderboard_scope", f"{target_col}_{split_name}_aggregate")
    agg_leaderboards.append(tmp)

aggregate_leaderboard_df = pd.concat(agg_leaderboards, ignore_index=True)

agg_metrics_df.to_csv(RUN_ROOT / "aggregate_metrics_by_model.csv", index=False)
agg_metrics_df.to_csv(TABLE_DIR / f"table_55_aggregate_metrics_by_model_{RUN_ID}.csv", index=False)

aggregate_leaderboard_df.to_csv(RUN_ROOT / "aggregate_leaderboard_by_model.csv", index=False)
aggregate_leaderboard_df.to_csv(TABLE_DIR / f"table_56_aggregate_leaderboard_by_model_{RUN_ID}.csv", index=False)

print("\nAggregate leaderboard, test split:")
print(
    aggregate_leaderboard_df[
        aggregate_leaderboard_df["split"] == "test"
    ][[
        "target_col",
        "model_name",
        "model_family",
        "mean_accuracy",
        "mean_balanced_accuracy",
        "mean_macro_f1",
        "mean_quadratic_weighted_kappa",
        "mean_ordinal_mae",
        "mean_ece_10bin",
        "aggregate_composite_rank",
    ]].to_string(index=False)
)

# ============================================================
# 12. Select models for downstream allocation
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Selecting validation-based models for downstream allocation")
print("=" * 80)

selected_rows = []

for target_col in TARGET_COLS:
    tmp = aggregate_leaderboard_df[
        (aggregate_leaderboard_df["target_col"] == target_col)
        & (aggregate_leaderboard_df["split"] == "validation")
    ].copy()

    if tmp.empty:
        continue

    best = tmp.sort_values("aggregate_composite_rank", ascending=True).iloc[0]

    selected_rows.append({
        "run_id": RUN_ID,
        "target_col": target_col,
        "selected_by": "aggregate_validation_composite_rank_purged_walk_forward",
        "selected_model": best["model_name"],
        "model_family": best["model_family"],
        "validation_aggregate_composite_rank": float(best["aggregate_composite_rank"]),
        "validation_mean_macro_f1": float(best["mean_macro_f1"]),
        "validation_mean_balanced_accuracy": float(best["mean_balanced_accuracy"]),
        "validation_mean_qwk": float(best["mean_quadratic_weighted_kappa"]),
        "validation_mean_ordinal_mae": float(best["mean_ordinal_mae"]),
        "validation_mean_ece": float(best["mean_ece_10bin"]),
    })

selected_models_df = pd.DataFrame(selected_rows)
selected_models_df.to_csv(RUN_ROOT / "selected_models_for_allocation_purged_walk_forward.csv", index=False)
selected_models_df.to_csv(TABLE_DIR / f"table_57_selected_models_for_allocation_purged_walk_forward_{RUN_ID}.csv", index=False)

print(selected_models_df.to_string(index=False))

# Create allocation input index for later notebooks.
allocation_index_rows = []

for _, row in selected_models_df.iterrows():
    target_col = row["target_col"]
    model_name = row["selected_model"]

    # Save model-specific probabilities and predictions from aggregate output.
    pred_model = predictions_all_df[
        (predictions_all_df["target_col"] == target_col)
        & (predictions_all_df["model_name"] == model_name)
    ].copy()

    proba_model = probabilities_all_df[
        (probabilities_all_df["target_col"] == target_col)
        & (probabilities_all_df["model_name"] == model_name)
    ].copy()

    pred_path_parquet = PRED_DIR / f"selected_predictions_{safe_name(target_col)}_{safe_name(model_name)}.parquet"
    pred_path_csv = PRED_DIR / f"selected_predictions_{safe_name(target_col)}_{safe_name(model_name)}.csv"

    proba_path_parquet = PROBA_DIR / f"selected_probabilities_{safe_name(target_col)}_{safe_name(model_name)}.parquet"
    proba_path_csv = PROBA_DIR / f"selected_probabilities_{safe_name(target_col)}_{safe_name(model_name)}.csv"

    pred_model.to_parquet(pred_path_parquet)
    pred_model.to_csv(pred_path_csv)

    proba_model.to_parquet(proba_path_parquet)
    proba_model.to_csv(proba_path_csv)

    role = "primary_20d" if "20d" in target_col else "primary_60d"

    allocation_index_rows.append({
        "run_id": RUN_ID,
        "policy_name": "P0_purged_walk_forward_validation_selected",
        "target_col": target_col,
        "role": role,
        "model_name": model_name,
        "model_family": row["model_family"],
        "probability_path_parquet": str(proba_path_parquet),
        "probability_path_csv": str(proba_path_csv),
        "prediction_path_parquet": str(pred_path_parquet),
        "prediction_path_csv": str(pred_path_csv),
        "source": "purged_walk_forward_validation_selected",
    })

allocation_input_index_df = pd.DataFrame(allocation_index_rows)
allocation_input_index_df.to_csv(RUN_ROOT / "NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv", index=False)
allocation_input_index_df.to_csv(TABLE_DIR / f"table_58_allocation_input_index_purged_wf_{RUN_ID}.csv", index=False)

print("\nAllocation input index:")
print(allocation_input_index_df.to_string(index=False))

# ============================================================
# 13. Plots
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Creating diagnostic plots")
print("=" * 80)

def plot_aggregate_metric(target_col, split_name, metric_col, path):
    tmp = aggregate_leaderboard_df[
        (aggregate_leaderboard_df["target_col"] == target_col)
        & (aggregate_leaderboard_df["split"] == split_name)
    ].copy()

    if tmp.empty:
        return

    tmp = tmp.sort_values(metric_col, ascending=False)

    plt.figure(figsize=(11, max(4, 0.35 * len(tmp))))
    sns.barplot(data=tmp, y="model_name", x=metric_col, color="#4C72B0")
    plt.title(f"{target_col} {split_name}: {metric_col}")
    plt.xlabel(metric_col)
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

for target_col in TARGET_COLS:
    for split_name in ["validation", "test"]:
        for metric_col in ["mean_macro_f1", "mean_balanced_accuracy", "mean_quadratic_weighted_kappa"]:
            plot_aggregate_metric(
                target_col=target_col,
                split_name=split_name,
                metric_col=metric_col,
                path=PLOT_DIR / f"{safe_name(target_col)}_{split_name}_{metric_col}.png",
            )

def plot_reliability_curve(y_true, proba, path, title, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    proba = normalize_proba(proba)

    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y_true).astype(float)

    bins = np.linspace(0, 1, n_bins + 1)
    bin_centers = []
    bin_acc = []
    bin_conf = []
    bin_counts = []

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)

        if mask.sum() == 0:
            continue

        bin_centers.append((lo + hi) / 2)
        bin_acc.append(correct[mask].mean())
        bin_conf.append(conf[mask].mean())
        bin_counts.append(mask.sum())

    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
    plt.plot(bin_conf, bin_acc, marker="o", label="Observed")
    plt.title(title)
    plt.xlabel("Mean confidence")
    plt.ylabel("Accuracy")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

# Reliability plots for selected models on test folds pooled.
for _, row in selected_models_df.iterrows():
    target_col = row["target_col"]
    model_name = row["selected_model"]

    pred_sel = predictions_all_df[
        (predictions_all_df["target_col"] == target_col)
        & (predictions_all_df["model_name"] == model_name)
        & (predictions_all_df["split"] == "test")
    ].copy()

    proba_sel = probabilities_all_df[
        (probabilities_all_df["target_col"] == target_col)
        & (probabilities_all_df["model_name"] == model_name)
        & (probabilities_all_df["split"] == "test")
    ].copy()

    if pred_sel.empty or proba_sel.empty:
        continue

    p = proba_sel[proba_cols()].values
    y = pred_sel["y_true"].values

    plot_reliability_curve(
        y_true=y,
        proba=p,
        path=PLOT_DIR / f"reliability_test_{safe_name(target_col)}_{safe_name(model_name)}.png",
        title=f"Reliability: {target_col}, {model_name}, test folds",
    )

# Copy selected plots to global figures.
for p in sorted(PLOT_DIR.glob("*.png")):
    target = FIGURE_DIR / f"{p.stem}_{RUN_ID}.png"
    target.write_bytes(p.read_bytes())

# ============================================================
# 14. Validation report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 8: Saving validation report and manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "07_AURORA_purged_embargoed_walk_forward_models.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "model_data_path": str(MODEL_DATA_PATH),
    "run_root": str(RUN_ROOT),
    "target_cols": TARGET_COLS,
    "target_horizon_map": TARGET_HORIZON_MAP,
    "n_features": len(feature_cols),
    "dataset_shape": list(df.shape),
    "date_start": str(df.index.min().date()),
    "date_end": str(df.index.max().date()),
    "walk_forward_folds": WALK_FORWARD_FOLDS,
    "min_train_size": MIN_TRAIN_SIZE,
    "min_val_size": MIN_VAL_SIZE,
    "min_test_size": MIN_TEST_SIZE,
    "purging_and_embargo": {
        "description": "Target-specific embargo is applied between train-validation and validation-test boundaries.",
        "20d_embargo_rows": TARGET_HORIZON_MAP["TAIEX_regime_fixed_20d"],
        "60d_embargo_rows": TARGET_HORIZON_MAP["TAIEX_regime_fixed_60d"],
        "previous_label_persistence_included": False,
    },
    "model_families": sorted(metrics_df["model_family"].dropna().unique().tolist()),
    "n_metric_rows": int(len(metrics_df)),
    "n_prediction_rows": int(len(predictions_all_df)),
    "n_probability_rows": int(len(probabilities_all_df)),
    "selected_models": selected_models_df.to_dict(orient="records"),
    "output_paths": {
        "metrics": str(metrics_path),
        "predictions": str(PRED_DIR / "predictions_all_purged_walk_forward_models.parquet"),
        "probabilities": str(PROBA_DIR / "probabilities_all_purged_walk_forward_models.parquet"),
        "aggregate_leaderboard": str(RUN_ROOT / "aggregate_leaderboard_by_model.csv"),
        "selected_models": str(RUN_ROOT / "selected_models_for_allocation_purged_walk_forward.csv"),
        "allocation_input_index": str(RUN_ROOT / "NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv"),
    },
}

validation_report_path = REPORT_RUN_DIR / "AURORA_07_purged_walk_forward_validation_report.json"
validation_report_global_path = REPORT_DIR / f"AURORA_07_purged_walk_forward_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "AURORA_07_purged_walk_forward_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"AURORA_07_purged_walk_forward_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 15. Final summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF NOTEBOOK 07 COMPLETE")
print("=" * 80)
print("Run ID                         :", RUN_ID)
print("Run root                       :", RUN_ROOT)
print("Split report                   :", TABLE_DIR / f"table_47_purged_walk_forward_split_report_{RUN_ID}.csv")
print("Metrics table                  :", TABLE_DIR / f"table_48_purged_walk_forward_metrics_all_{RUN_ID}.csv")
print("Aggregate metrics              :", TABLE_DIR / f"table_55_aggregate_metrics_by_model_{RUN_ID}.csv")
print("Aggregate leaderboard          :", TABLE_DIR / f"table_56_aggregate_leaderboard_by_model_{RUN_ID}.csv")
print("Selected models                :", TABLE_DIR / f"table_57_selected_models_for_allocation_purged_walk_forward_{RUN_ID}.csv")
print("Allocation input index         :", RUN_ROOT / "NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv")
print("Predictions                    :", PRED_DIR / "predictions_all_purged_walk_forward_models.parquet")
print("Probabilities                  :", PROBA_DIR / "probabilities_all_purged_walk_forward_models.parquet")
print("Plots                          :", PLOT_DIR)
print("Validation report              :", validation_report_path)
print("Manifest                       :", manifest_path)
print("=" * 80)

print("\nRecommended next notebook:")
print("08_AURORA_stronger_financial_baselines_and_purged_allocation.ipynb")

Mounted at /content/drive
AURORA-TWETF Notebook 07: Purged / Embargoed Walk-Forward Models
Timestamp UTC : 2026-06-24T03:18:17Z
Run ID        : 20260624_031817
Project root  : /content/drive/MyDrive/AURORA_TWETF
Input data    : /content/drive/MyDrive/AURORA_TWETF/data/modeling/AURORA_TWETF_features_with_labels.parquet
Run root      : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817

Step 1: Loading modeling dataset
Dataset shape        : (1262, 417)
Feature columns      : 415
Date range           : 2021-01-06 to 2026-03-25

Target distribution: TAIEX_regime_fixed_20d
TAIEX_regime_fixed_20d
0     40
1    196
2    536
3    429
4     61

Target distribution: TAIEX_regime_fixed_60d
TAIEX_regime_fixed_60d
0     91
1    191
2    270
3    314
4    396

Purged walk-forward split report:
            target_col fold_id  horizon  raw_train_end_index  raw_val_end_index  raw_test_end_index  train_start_index  train_end_index  val_start_index  v